# Module 5 Lab 3: Age, income, vehicles (Problem 5.5)
**Name:** Roey Kadur  
**GitHub repo:** https://github.com/roee0910/cs82a-portfolio (folder `module5/`)

In [1]:
import pandas as pd
from IPython.display import display

cust = pd.read_csv("custdata.tsv", sep="\t")
cols = ["age", "income", "num.vehicles"]
print(cust.shape)
cust[cols].describe().round(1)

(1000, 11)


,age,income,num.vehicles
count,1000.0,1000.0,944.0
mean,51.7,53504.8,1.9
std,18.9,65478.1,1.1
min,0.0,-8700.0,0.0
25%,38.0,14600.0,1.0
50%,50.0,35000.0,2.0
75%,64.0,67000.0,2.0
max,146.7,615000.0,6.0


Looking at the min and max, some values look wrong: the biggest age is about 147 and the smallest is 0, and the lowest income is negative.

## Step 1: BEFORE cleaning (the wrong answer on purpose)

In [2]:
before = cust[cols].corr()
before.round(3)

,age,income,num.vehicles
age,1.000,0.027,-0.064
income,0.027,1.000,0.139
num.vehicles,-0.064,0.139,1.000


## Step 2: Find the invalid values

In [3]:
bad_age = (cust["age"] <= 0) | (cust["age"] >= 100)
bad_income = cust["income"] < 0

print("Ages that are 0 or 100+:", bad_age.sum())
display(cust.loc[bad_age, ["custid", "age"]].sort_values("age").round(1))
print()
print("Negative incomes:", bad_income.sum())
display(cust.loc[bad_income, ["custid", "income"]])
print()
print("Income of exactly 0:", (cust["income"] == 0).sum())
print("Missing num.vehicles:", cust["num.vehicles"].isna().sum())

Ages that are 0 or 100+: 11


,custid,age
868,1229417,0.0
667,931449,0.0
965,1375846,0.0
953,1361012,113.3
266,364991,123.1
625,875771,124.8
926,1303929,126.7
752,1047534,130.4
286,397230,136.1
212,287882,137.7



Negative incomes: 1


,custid,income
691,971703,-8700



Income of exactly 0: 78
Missing num.vehicles: 56


**What I am removing and why:**

- **Age of 0 or 100 and over (11 rows):** a customer cannot be 0 years old, and ages like 123 or 147 are not realistic, so these are data errors. The oldest real-looking age left is 93.
- **Negative income (1 row):** income of -8700 does not make sense for this column.
- **I am keeping income = 0 (78 rows):** zero income is possible (for example someone who is retired or not working), so it is not clearly an error. Removing it would throw away real customers.
- **Missing num.vehicles (56 rows):** these are missing, not invalid, so I did not delete them. pandas leaves them out only when it calculates the correlations that use vehicles.

## Step 3: Remove the invalid rows and count

In [4]:
clean = cust[~bad_age & ~bad_income]
print("Rows before:", len(cust))
print("Rows after:", len(clean))
print("Rows removed:", len(cust) - len(clean))

# self-check: the numbers add up and nothing invalid is left
assert len(cust) - len(clean) == bad_age.sum() + bad_income.sum()
assert clean["age"].between(1, 99).all()
assert (clean["income"] >= 0).all()

Rows before: 1000
Rows after: 988
Rows removed: 12


## Step 4: AFTER cleaning

In [5]:
after = clean[cols].corr()
after.round(3)

,age,income,num.vehicles
age,1.000,0.009,-0.066
income,0.009,1.000,0.138
num.vehicles,-0.066,0.138,1.000


## Step 5: Before vs after side by side

In [6]:
print("BEFORE:")
display(before.round(2))
print()
print("AFTER:")
display(after.round(2))
print()
print("Change (after minus before):")
display((after - before).round(3))

BEFORE:


,age,income,num.vehicles
age,1.00,0.03,-0.06
income,0.03,1.00,0.14
num.vehicles,-0.06,0.14,1.00



AFTER:


,age,income,num.vehicles
age,1.00,0.01,-0.07
income,0.01,1.00,0.14
num.vehicles,-0.07,0.14,1.00



Change (after minus before):


,age,income,num.vehicles
age,0.000,-0.019,-0.003
income,-0.019,0.000,-0.001
num.vehicles,-0.003,-0.001,0.000


**Honest note about the contrast:** the two matrices are very close. The biggest change is age vs income (0.027 before, 0.009 after), and income vs vehicles stays about 0.14. So in this data set the bad values did not change the answer much, but I still removed them because impossible ages and a negative income are errors, and I could not know that ahead of time.

## Interpretation (cleaned result only)

After cleaning, all three correlations are weak. Age and income are basically unrelated (r = 0.01), and age and number of vehicles is a tiny negative (r = -0.07), which is almost nothing. The only one that stands out a little is income and number of vehicles (r = 0.14): people with higher income tend to own slightly more vehicles, but it is a weak link (r squared is only about 0.02, so income explains about 2% of the differences). One caution is that this shows a relationship, not a cause, and the vehicles numbers use 932 rows because 56 customers had no vehicle data.